# Gradio App and Deployment
**Author:** Juan Esteban Agudelo Ortiz  
**Email:** juan.es.agor@gmail.com

---

This notebook packages the full RAG pipeline into a Gradio web application
and deploys it as a HuggingFace Space. The app accepts PDFs, handwritten
note images, and reference images as input, and generates structured flash
cards or consolidated summaries exportable as PDF.

The app is designed for students without programming knowledge. All model
downloads, indexing, and inference happen automatically on first run.

### Architecture
The app consolidates the full pipeline into a single `src/` module:

- `src/ingestion.py`   — document ingestion and OCR (notebooks 1-2)
- `src/chunking.py`    — chunking strategies (notebook 3)
- `src/indexing.py`    — embeddings and vector store (notebook 4)
- `src/generation.py`  — RAG pipeline and structured output (notebook 5)
- `src/export.py`      — PDF export (this notebook)
- `app.py`             — Gradio interface and entry point

### Limitations
1. First run downloads the model (~2.5GB) and builds the index; expect
   5-10 minutes before the app is ready.
2. Generation is slow on CPU; expect 1-3 minutes per flash card.
3. Equation recognition is experimental; manual region selection is
   recommended (see notebook 2).
4. Context precision is limited by the retriever; complex multi-topic
   queries may produce incomplete flash cards.

## 0. Install dependencies

In [1]:
# Run only once
# !pip install gradio pymupdf paddleocr paddlepaddle pillow numpy \
#             llama-cpp-python llama-index-core \
#             llama-index-embeddings-huggingface \
#             llama-index-vector-stores-chroma chromadb \
#             weasyprint huggingface-hub

## 1. Imports and configuration

In [2]:
import os
import gc
import json
import re
import tempfile
from pathlib import Path
from dataclasses import dataclass, field
from typing import Optional
from enum import Enum

import gradio as gr
from huggingface_hub import hf_hub_download

# --- Directory setup ---
BASE_DIR    = Path("..")
DATA_DIR    = BASE_DIR / "data"
MODELS_DIR  = DATA_DIR / "models"
INDEX_DIR   = DATA_DIR / "index"
UPLOADS_DIR = DATA_DIR / "uploads"
OUT_DIR     = DATA_DIR / "outputs"

for d in [MODELS_DIR, INDEX_DIR, UPLOADS_DIR, OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# --- Model constants ---
EMBEDDING_MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
GENERATOR_MODEL      = "qwen2.5-3b-instruct-q4_k_m.gguf"
GENERATOR_PATH       = MODELS_DIR / GENERATOR_MODEL

# --- Output modes ---
VALID_MODES = {"flashcard", "summary"}

print("Configuration ready ✓")

Configuration ready ✓


## 2. Pipeline functions

This section consolidates the core pipeline functions from the previous
notebooks into a single module. In the deployed app these functions live
in `src/` and are imported by `app.py`. Here they are defined inline
for notebook demonstration purposes.

The pipeline has four stages:

1. **Ingestion:** extract text from PDFs and images
2. **Indexing:** chunk text, embed, and store in ChromaDB
3. **Generation:** retrieve context and generate structured output
4. **Export:** convert structured output to PDF

In [3]:
import fitz
import numpy as np
import chromadb
from llama_cpp import Llama
from llama_index.core import VectorStoreIndex, StorageContext, Document
from llama_index.core.node_parser import SentenceSplitter
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore

# ── Input types ──────────────────────────────────────────────────────────────

class InputType(Enum):
    PDF           = "pdf"
    HANDWRITTEN   = "handwritten_image"
    REFERENCE_IMG = "reference_image"
    PLAIN_TEXT    = "plain_text"

@dataclass
class IngestedDocument:
    input_type       : InputType
    text             : str
    reference_images : list = field(default_factory=list)
    source_path      : Optional[Path] = None

@dataclass
class FlashCard:
    concept         : str
    definition      : str
    key_points      : list
    examples        : list
    suggested_image : str

@dataclass
class ConsolidatedSummary:
    topic    : str
    overview : str
    concepts : list

# ── Ingestion ─────────────────────────────────────────────────────────────────

def extract_text_from_pdf(pdf_path: Path, min_block_chars: int = 20) -> str:
    doc = fitz.open(pdf_path)
    all_text = []
    for page_num, page in enumerate(doc):
        blocks = page.get_text("blocks")
        blocks_sorted = sorted(blocks, key=lambda b: (b[1], b[0]))
        page_text = []
        for block in blocks_sorted:
            text = block[4].strip()
            if len(text) < min_block_chars:
                continue
            text = re.sub(r"\s+", " ", text)
            page_text.append(text)
        if page_text:
            all_text.append(f"--- Page {page_num + 1} ---\n" + "\n\n".join(page_text))
    doc.close()
    return "\n\n".join(all_text)

# ── Chunking ──────────────────────────────────────────────────────────────────

def fixed_size_chunking(
    text: str,
    chunk_size: int = 512,
    chunk_overlap: int = 64,
) -> list[dict]:
    splitter  = SentenceSplitter(chunk_size=chunk_size, chunk_overlap=chunk_overlap)
    llama_doc = Document(text=text)
    nodes     = splitter.get_nodes_from_documents([llama_doc])
    return [{"text": node.text, "index": i, "strategy": "fixed_size"} for i, node in enumerate(nodes)]

# ── Indexing ──────────────────────────────────────────────────────────────────

METADATA_FILE = "index_metadata.json"

def load_or_build_index(
    chunks: list[dict],
    embed_model: HuggingFaceEmbedding,
    collection_name: str,
    persist_dir: Path,
) -> VectorStoreIndex:
    metadata_path = persist_dir / METADATA_FILE
    if metadata_path.exists():
        stored = json.loads(metadata_path.read_text(encoding="utf-8"))
        if stored.get("embedding_model") == EMBEDDING_MODEL_NAME:
            _, vector_store = _create_vector_store(collection_name, persist_dir)
            return VectorStoreIndex.from_vector_store(vector_store, embed_model=embed_model)
        import shutil
        shutil.rmtree(persist_dir)
        persist_dir.mkdir(parents=True, exist_ok=True)

    _, vector_store = _create_vector_store(collection_name, persist_dir)
    documents = [
        Document(text=c["text"], metadata={"chunk_index": c["index"], "strategy": c["strategy"]})
        for c in chunks
    ]
    storage_context = StorageContext.from_defaults(vector_store=vector_store)
    index = VectorStoreIndex.from_documents(
        documents, storage_context=storage_context, embed_model=embed_model, show_progress=True
    )
    metadata_path.write_text(json.dumps({"embedding_model": EMBEDDING_MODEL_NAME}), encoding="utf-8")
    return index

def _create_vector_store(collection_name: str, persist_dir: Path):
    client     = chromadb.PersistentClient(path=str(persist_dir))
    collection = client.get_or_create_collection(collection_name)
    return collection, ChromaVectorStore(chroma_collection=collection)

# ── Generation ────────────────────────────────────────────────────────────────

SYSTEM_PROMPT_FLASHCARD = """You are an expert study assistant that creates structured flash cards for students.

Given a context extracted from a textbook and a topic query, generate a flash card in JSON format.

IMPORTANT: Always respond in the same language as the topic query. If they ask in Spanish, answer in Spanish. If they ask in English, answer in English.

The JSON must follow this exact schema with no additional fields:

{{
    "concept": "name of the concept",
    "definition": "clear and concise definition in 2-3 sentences",
    "key_points": ["point 1", "point 2", "point 3"],
    "examples": ["example 1", "example 2"],
    "suggested_image": "brief description of an image that would illustrate this concept"
}}

Rules:
- Respond ONLY with the JSON object, no preamble, no explanation, no markdown backticks
- Base your response strictly on the provided context
- If the context does not contain enough information, still follow the schema but indicate the limitation in the definition field
- All fields are required"""


SYSTEM_PROMPT_SUMMARY = """You are an expert study assistant that creates structured summaries for students.

Given a context extracted from a textbook and a topic query, generate a consolidated summary in JSON format.
IMPORTANT: Always respond in the same language as the topic query. If they ask in Spanish, answer in Spanish. If they ask in English, answer in English.

The JSON must follow this exact schema with no additional fields:

{{
    "topic": "main topic name",
    "overview": "2-3 sentence overview of the topic",
    "concepts": [
        {{
            "name": "concept name",
            "description": "brief description in 1-2 sentences"
        }}
    ]
}}

Rules:
- Respond ONLY with the JSON object, no preamble, no explanation, no markdown backticks
- Base your response strictly on the provided context
- Include between 3 and 8 concepts
- All fields are required"""

def generate_flashcard(
    query: str,
    index: VectorStoreIndex,
    llm: Llama,
    embed_model: HuggingFaceEmbedding,
    mode: str = "flashcard",
    k: int = 3,
) -> FlashCard | ConsolidatedSummary:
    if mode not in VALID_MODES:
        raise ValueError(f"Invalid mode '{mode}'. Must be one of {VALID_MODES}")

    system_prompt = SYSTEM_PROMPT_FLASHCARD if mode == "flashcard" else SYSTEM_PROMPT_SUMMARY
    example = FLASHCARD_EXAMPLE if mode == "flashcard" else SUMMARY_EXAMPLE

    retriever = index.as_retriever(similarity_top_k=k, embed_model=embed_model)
    nodes     = retriever.retrieve(query)
    if not nodes:
        raise ValueError(f"No chunks retrieved for query: '{query}'")

    context = "\n\n".join(f"[Chunk {i+1}]\n{node.text}" for i, node in enumerate(nodes))

    for attempt, prompt in enumerate([system_prompt, system_prompt + f"\n\nExample:\n{example}"]):
        try:
            response = llm.create_chat_completion(
                messages    = [
                    {"role": "system", "content": prompt},
                    {"role": "user",   "content": f"Context:\n{context}\n\nTopic: {query}"},
                ],
                max_tokens  = 512 if mode == "flashcard" else 768,
                temperature = 0.1,
            )
            raw  = json.loads(response["choices"][0]["message"]["content"].strip())
            if mode == "flashcard":
                return FlashCard(**raw)
            return ConsolidatedSummary(**raw)
        except (json.JSONDecodeError, KeyError, TypeError) as e:
            if attempt == 1:
                raise ValueError(f"Generation failed after 2 attempts: {e}")

print("Pipeline functions defined ✓")

Pipeline functions defined ✓


## 3. Flash card and summary generation

This section defines the constants used in generation and tests the
full pipeline end to end: ingest a PDF, build the index, and generate
a flash card and a consolidated summary for a chemistry topic.

In [4]:
FLASHCARD_EXAMPLE = """{
    "concept": "Amide",
    "definition": "An amide is a compound derived from a carboxylic acid where the hydroxyl group is replaced by an amino group. Amides are found in proteins and many biological molecules.",
    "key_points": [
        "Derived from carboxylic acids",
        "Contains a carbonyl group bonded to nitrogen",
        "Found in proteins as peptide bonds"
    ],
    "examples": ["Acetamide (CH3CONH2)", "Nylon (synthetic polyamide)"],
    "suggested_image": "Structural formula of an amide showing the carbonyl group bonded to nitrogen"
}"""

SUMMARY_EXAMPLE = """{
    "topic": "Carboxylic Acid Derivatives",
    "overview": "Carboxylic acid derivatives are compounds that can be hydrolyzed to give carboxylic acids.",
    "concepts": [
        {"name": "Ester", "description": "Formed by reaction of carboxylic acid with alcohol"},
        {"name": "Amide", "description": "Formed by reaction of carboxylic acid with amine"}
    ]
}"""

# --- Download model if not present ---
if not GENERATOR_PATH.exists():
    print(f"Downloading {GENERATOR_MODEL} (~2GB)...")
    hf_hub_download(
        repo_id   = "Qwen/Qwen2.5-3B-Instruct-GGUF",
        filename  = GENERATOR_MODEL,
        local_dir = str(MODELS_DIR),
    )
    print("Download complete ✓")
else:
    print(f"Model present: {GENERATOR_PATH}")

# --- Initialize pipeline components ---
embed_model = HuggingFaceEmbedding(
    model_name = EMBEDDING_MODEL_NAME,
    device     = "cpu",
)

llm = Llama(
    model_path = str(GENERATOR_PATH),
    n_ctx      = 2048,
    n_threads  = os.cpu_count(),
    verbose    = False,
)

# --- Test with OpenStax chapter 21 ---
pdf_path = UPLOADS_DIR / "openstax_ch21_carboxylic_acid_derivatives.pdf"
text     = extract_text_from_pdf(pdf_path)
chunks   = fixed_size_chunking(text)
index    = load_or_build_index(
    chunks          = chunks,
    embed_model     = embed_model,
    collection_name = "ch21_app",
    persist_dir     = INDEX_DIR / "ch21_app",
)

# --- Generate flash card ---
card = generate_flashcard("amida", index, llm, embed_model, mode="flashcard")
print(f"Concept    : {card.concept}")
print(f"Definition : {card.definition}")
print(f"Key points : {card.key_points}")

# --- Generate summary ---
summary = generate_flashcard("esters", index, llm, embed_model, mode="summary")
print(f"\nTopic    : {summary.topic}")
print(f"Overview : {summary.overview}")

Model present: ../data/models/qwen2.5-3b-instruct-q4_k_m.gguf


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized
/home/juanessao/Documents/Datos_de_Ciencia/playlists/hugging-face-hackaton/flashcard-generator-slm/.venv/lib/python3.13/site-packages/pydantic/_internal/_generate_schema.py:949: UserWarning: Mixing V1 models and V2 models (or constructs, like `TypeAdapter`) is not supported. Please upgrade `BasePromptTemplate` to V2.
  warnings.warn(


Concept    : Amida
Definition : Amida es una clase de compuestos derivados del ácido carboxílico donde el grupo hidroxilo es reemplazado por un grupo amino. Los amidas son encontrados en proteínas y muchos moléculas biológicamente importantes.
Key points : ['Derivado del ácido carboxílico', 'Contiene un grupo carbólico unido a nitrógeno', 'Presente en proteínas como enlaces peptídicos']

Topic    : esters
Overview : Esters are a class of compounds that are widely used in the chemical industry and in nature. They are formed by the reaction of carboxylic acids with alcohols and are important in both biological and industrial applications.


## 4. PDF export

Flash cards and consolidated summaries are exported as PDF using
WeasyPrint, which renders HTML/CSS to PDF. The flash card layout
has two columns: a reference image on the left and structured
content on the right.

WeasyPrint converts an HTML string to a PDF file in three steps:
1. Parse the HTML and CSS
2. Lay out the document into pages
3. Render each page to PDF

The output is a single PDF file saved to the outputs directory.

In [5]:
from weasyprint import HTML
from PIL import Image
import base64
import io


def image_to_base64(img: Image.Image) -> str:
    """
    Convert a PIL Image to a base64 string for embedding in HTML.

    Parameters
    ----------
    img : PIL.Image.Image
        Image to convert.

    Returns
    -------
    str
        Base64 encoded image string with data URI prefix.
    """
    buffer = io.BytesIO()
    img.save(buffer, format="PNG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return f"data:image/png;base64,{encoded}"


def flashcard_to_html(card: FlashCard, reference_image: Image.Image = None) -> str:
    """
    Convert a FlashCard to an HTML string for PDF rendering.

    Parameters
    ----------
    card : FlashCard
        Structured flash card content.
    reference_image : PIL.Image.Image, optional
        Reference image to embed in the card.

    Returns
    -------
    str
        HTML string ready for WeasyPrint rendering.
    """
    key_points_html = "".join(f"<li>{p}</li>" for p in card.key_points)
    examples_html   = "".join(f"<li>{e}</li>" for e in card.examples)

    img_html = ""
    if reference_image is not None:
        img_src  = image_to_base64(reference_image)
        img_html = f'<img src="{img_src}" style="max-width:100%; border-radius:8px;">'
    else:
        img_html = f'<p style="color:#888; font-style:italic;">{card.suggested_image}</p>'

    return f"""
    <html>
    <head>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
            background: #ffffff;
        }}
        .card {{
            display: flex;
            border: 2px solid #2ecc71;
            border-radius: 12px;
            padding: 20px;
            gap: 20px;
        }}
        .image-col {{
            width: 35%;
            display: flex;
            align-items: center;
            justify-content: center;
        }}
        .content-col {{
            width: 65%;
        }}
        h1 {{
            color: #2ecc71;
            font-size: 22px;
            margin-bottom: 8px;
        }}
        h3 {{
            color: #555;
            font-size: 14px;
            margin: 12px 0 4px 0;
        }}
        ul {{
            margin: 4px 0;
            padding-left: 20px;
        }}
        li {{
            font-size: 13px;
            margin-bottom: 3px;
        }}
        p {{
            font-size: 13px;
            line-height: 1.5;
        }}
    </style>
    </head>
    <body>
        <div class="card">
            <div class="image-col">{img_html}</div>
            <div class="content-col">
                <h1>{card.concept}</h1>
                <p>{card.definition}</p>
                <h3>Key Points</h3>
                <ul>{key_points_html}</ul>
                <h3>Examples</h3>
                <ul>{examples_html}</ul>
            </div>
        </div>
    </body>
    </html>
    """


def summary_to_html(summary: ConsolidatedSummary) -> str:
    """
    Convert a ConsolidatedSummary to an HTML string for PDF rendering.

    Parameters
    ----------
    summary : ConsolidatedSummary
        Structured summary content.

    Returns
    -------
    str
        HTML string ready for WeasyPrint rendering.
    """
    concepts_html = "".join(
        f"<li><strong>{c['name']}:</strong> {c['description']}</li>"
        for c in summary.concepts
    )

    return f"""
    <html>
    <head>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 20px;
            background: #ffffff;
        }}
        .summary {{
            border: 2px solid #3498db;
            border-radius: 12px;
            padding: 20px;
        }}
        h1 {{
            color: #3498db;
            font-size: 22px;
            margin-bottom: 8px;
        }}
        h3 {{
            color: #555;
            font-size: 14px;
            margin: 12px 0 4px 0;
        }}
        ul {{
            margin: 4px 0;
            padding-left: 20px;
        }}
        li {{
            font-size: 13px;
            margin-bottom: 6px;
        }}
        p {{
            font-size: 13px;
            line-height: 1.5;
        }}
    </style>
    </head>
    <body>
        <div class="summary">
            <h1>{summary.topic}</h1>
            <p>{summary.overview}</p>
            <h3>Concepts</h3>
            <ul>{concepts_html}</ul>
        </div>
    </body>
    </html>
    """


def export_to_pdf(
    content: FlashCard | ConsolidatedSummary,
    output_path: Path,
    reference_image: Image.Image = None,
) -> Path:
    """
    Export a FlashCard or ConsolidatedSummary to PDF.

    Parameters
    ----------
    content : FlashCard or ConsolidatedSummary
        Content to export.
    output_path : Path
        Path where the PDF will be saved.
    reference_image : PIL.Image.Image, optional
        Reference image for flash cards.

    Returns
    -------
    Path
        Path to the generated PDF file.
    """
    if isinstance(content, FlashCard):
        html = flashcard_to_html(content, reference_image)
    else:
        html = summary_to_html(content)

    HTML(string=html).write_pdf(str(output_path))
    return output_path


# --- Test export ---
pdf_path_card    = OUT_DIR / "test_flashcard.pdf"
pdf_path_summary = OUT_DIR / "test_summary.pdf"

export_to_pdf(card, pdf_path_card)
export_to_pdf(summary, pdf_path_summary)

print(f"Flash card PDF : {pdf_path_card}")
print(f"Summary PDF    : {pdf_path_summary}")

Flash card PDF : ../data/outputs/test_flashcard.pdf
Summary PDF    : ../data/outputs/test_summary.pdf


## 5. Gradio interface

The Gradio interface has three tabs:

- **Flash Card:** generates a single concept card from uploaded documents
- **Summary:** generates a consolidated summary of multiple concepts
- **About:** instructions for use

Input components:
- PDF upload (optional)
- Handwritten notes image upload (optional)
- Reference image upload (optional)
- Topic text input (required)
- Page range sliders for PDF chapter extraction
- Output mode selector (flash card / summary)

The model and index are initialized once at app startup and reused
across all user interactions.

> **Note:** The interface defined above is for demonstration purposes only.
> Running Gradio inside a Jupyter notebook has known limitations with file
> handling and port management. The deployable version lives in `app.py`
> at the repository root and must be run from the terminal with
> `python app.py`.

In [6]:
import gradio as gr

# ── Global initialization (runs once at startup) ──────────────────────────────

print("Initializing pipeline components...")

embed_model_global = HuggingFaceEmbedding(
    model_name = EMBEDDING_MODEL_NAME,
    device     = "cpu",
)

llm_global = Llama(
    model_path = str(GENERATOR_PATH),
    n_ctx      = 2048,
    n_threads  = os.cpu_count(),
    verbose    = False,
)

print("Pipeline ready ✓")


# ── Helper: build index from uploaded files ───────────────────────────────────

def process_inputs(
    pdf_file,
    notes_image,
    ref_image,
    page_start: int,
    page_end: int,
) -> tuple[str, list]:
    """
    Process uploaded files and return extracted text and reference images.

    Parameters
    ----------
    pdf_file : str or None
        Path to uploaded PDF file.
    notes_image : str or None
        Path to uploaded handwritten notes image.
    ref_image : str or None
        Path to uploaded reference image.
    page_start : int
        First page to extract from PDF (1-indexed).
    page_end : int
        Last page to extract from PDF (1-indexed).

    Returns
    -------
    tuple[str, list]
        Extracted text and list of PIL reference images.
    """
    from paddleocr import PaddleOCR
    from PIL import Image as PILImage
    import fitz

    text_parts     = []
    reference_imgs = []

    if pdf_file is not None:
        doc = fitz.open(pdf_file)
        total_pages = len(doc)
        start = max(0, page_start - 1)
        end   = min(total_pages, page_end)

        sub = fitz.open()
        sub.insert_pdf(doc, from_page=start, to_page=end - 1)
        tmp_path = Path(tempfile.mktemp(suffix=".pdf"))
        sub.save(str(tmp_path))
        doc.close()

        text_parts.append(extract_text_from_pdf(tmp_path))
        tmp_path.unlink()

    if notes_image is not None:
        ocr = PaddleOCR(use_angle_cls=True, lang="es")
        result = ocr.ocr(notes_image, cls=True)
        if result and result[0]:
            lines = [line[1][0] for line in result[0] if line[1][1] >= 0.7]
            text_parts.append("\n".join(lines))

    if ref_image is not None:
        img = PILImage.open(ref_image).convert("RGB")
        reference_imgs.append(img)

    return "\n\n".join(text_parts), reference_imgs


# ── Callback: generate flash card ─────────────────────────────────────────────

def generate_card_callback(
    pdf_file,
    notes_image,
    ref_image,
    topic: str,
    page_start: int,
    page_end: int,
    mode: str,
) -> tuple[str, str]:
    """
    Gradio callback for flash card and summary generation.

    Returns
    -------
    tuple[str, str]
        (status message, path to generated PDF or empty string)
    """
    if not topic.strip():
        return "Please enter a topic.", ""

    try:
        text, ref_imgs = process_inputs(
            pdf_file, notes_image, ref_image, page_start, page_end
        )

        if not text.strip():
            return "No text could be extracted from the provided files.", ""

        chunks = fixed_size_chunking(text)
        index  = load_or_build_index(
            chunks          = chunks,
            embed_model     = embed_model_global,
            collection_name = f"session_{hash(text[:100])}",
            persist_dir     = INDEX_DIR / f"session_{hash(text[:100])}",
        )

        gradio_mode = "flashcard" if mode == "Flash Card" else "summary"
        result = generate_flashcard(
            query       = topic,
            index       = index,
            llm         = llm_global,
            embed_model = embed_model_global,
            mode        = gradio_mode,
        )

        ref_image_pil = ref_imgs[0] if ref_imgs else None
        pdf_out = OUT_DIR.resolve() / f"{topic.replace(' ', '_')}_{gradio_mode}.pdf"
        export_to_pdf(result, pdf_out, reference_image=ref_image_pil)

        return f"Generated successfully: {result.concept if gradio_mode == 'flashcard' else result.topic}", str(pdf_out)

    except Exception as e:
        return f"Error: {str(e)}", ""


print("Callbacks defined ✓")

Initializing pipeline components...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

llama_context: n_ctx_seq (2048) < n_ctx_train (32768) -- the full capacity of the model will not be utilized


Pipeline ready ✓
Callbacks defined ✓


In [7]:
# ── Gradio interface ───────────────────────────────────────────────────────────

with gr.Blocks(title="Flashcard Generator") as demo:

    gr.Markdown("""
    # 📚 Flashcard Generator
    Generate structured study flash cards from your documents using a local AI model.
    Upload a PDF, handwritten notes, or a reference image, enter a topic, and get a
    downloadable flash card or consolidated summary.
    """)

    with gr.Tab("Generate"):
        with gr.Row():
            with gr.Column(scale=1):
                pdf_input    = gr.File(label="PDF document (optional)", file_types=[".pdf"])
                page_start   = gr.Slider(minimum=1, maximum=500, value=1,  step=1, label="First page")
                page_end     = gr.Slider(minimum=1, maximum=500, value=50, step=1, label="Last page")
                notes_input  = gr.Image(label="Handwritten notes (optional)", type="filepath")
                ref_input    = gr.Image(label="Reference image (optional)", type="filepath")
                topic_input  = gr.Textbox(label="Topic", placeholder="e.g. amide, photosynthesis, Newton's laws")
                mode_input   = gr.Radio(
                    choices = ["Flash Card", "Summary"],
                    value   = "Flash Card",
                    label   = "Output mode"
                )
                submit_btn   = gr.Button("Generate", variant="primary")

            with gr.Column(scale=1):
                status_output = gr.Textbox(label="Status", interactive=False)
                pdf_output    = gr.File(label="Download PDF")

        submit_btn.click(
            fn      = generate_card_callback,
            inputs  = [pdf_input, notes_input, ref_input, topic_input, page_start, page_end, mode_input],
            outputs = [status_output, pdf_output],
        )

    with gr.Tab("About"):
        gr.Markdown("""
        ## How to use

        1. Upload a PDF, handwritten notes image, or reference image (at least one required)
        2. If uploading a PDF, use the page sliders to select the relevant chapter
        3. Enter the topic you want to study
        4. Select Flash Card for a single concept or Summary for multiple concepts
        5. Click Generate and download your PDF

        ## Limitations
        - Generation takes 1-3 minutes on CPU
        - Equation recognition is not supported in this version
        - Context precision may be limited for complex multi-topic queries

        ## Model
        Running locally with Qwen2.5-3B-Instruct (GGUF Q4_K_M) via llama.cpp.
        No data is sent to any external server.
        """)


print("Gradio interface defined ✓")

Gradio interface defined ✓


In [8]:
# Local deployment
# Uncomment to run locally:

demo.launch(
    server_name = "0.0.0.0",  # accessible from other devices on the network
    server_port = 7860,
    share       = False,       # set True for temporary public URL
)

print("To run locally, uncomment demo.launch() above.")
print("Or run: python app.py from the repository root.")

* Running on local URL:  http://0.0.0.0:7860
* To create a public link, set `share=True` in `launch()`.


To run locally, uncomment demo.launch() above.
Or run: python app.py from the repository root.


## 6. Local deployment

The app runs from the repository root via:

```text
python app.py
```

This requires the `src/` module and `app.py` to be present at the root
level. The notebook demonstrates the pipeline components — the actual
deployable app lives in `app.py` which imports from `src/`.

For Windows users, `run.bat` wraps this command so the app launches
with a double click.

> **Note:** Running the Gradio interface defined in Section 5 directly
> inside a Jupyter notebook has known limitations with file handling.
> Always run the app via `python app.py` from the terminal.

## 7. HuggingFace Spaces deployment

A HuggingFace Space is a free hosted server that runs a Gradio app
publicly. The Space reads a YAML header in `README.md` to configure
the runtime, installs dependencies from `requirements.txt`, and
executes `app.py` as the entry point.

The minimum files required for a Gradio Space:

```text
app.py              ← entry point, must be named exactly this
requirements.txt    ← dependencies installed automatically by HF
README.md           ← YAML header configures the Space
```

The `README.md` header must contain:

```yaml
---
title: Flashcard Generator SLM
emoji: 📚
colorFrom: green
colorTo: blue
sdk: gradio
sdk_version: 5.0.0
app_file: app.py
pinned: false
---
```

To deploy:

1. Create a new Space at `https://huggingface.co/new-space` and select Gradio as SDK
2. Add the Space as a remote and push:

```text
git remote add space https://huggingface.co/spaces/DatosDeCiencia-LAT/flashcard-generator-slm
git push space main
```

The Space installs `requirements.txt` automatically and runs `app.py`.
First run downloads the model (~2.5GB) — expect 5-10 minutes before
the app is ready.